[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q1/02_core_overlap.ipynb)

# R1-Q1 Week 3 — Core Overlap

You're in Week 3 of R1-Q1. Last week you ran differential expression on AtGenExpress and produced a long-format DE table covering 16 (stress, tissue) pairs. This week takes those DE flags and intersects them across stresses to identify the *common stress core* — genes that respond to a supermajority of the eight abiotic stresses, regardless of tissue. The core then goes into functional enrichment.

**By the end of this notebook you will have:**

- A per-gene tally of how many stresses it was DE in.
- A diagnostic view of gene counts versus the stress-count threshold N.
- The common stress core: genes DE in at least 6 of 8 stresses (either tissue).
- Functional enrichment results characterizing what the core is enriched for.
- A saved core gene set written to Drive for Notebook 4 to consume.

In [ ]:
# Setup. Three steps: install the iRI Lab library, mount Drive,
# then read the DE table that last week's notebook wrote to Drive.

# 1. Install the library (not pre-installed on Colab).
#!pip install git+https://github.com/geraldmc/irilab2026.git@main --quiet
!pip install --upgrade --force-reinstall --no-cache-dir git+https://github.com/geraldmc/irilab2026.git@main --quiet

# 2. Mount Drive.
import irilab2026 as iri
iri.setup()

# 3. Read the DE table from Drive — written by 01_deg_analysis.ipynb.
import pandas as pd
de_results = pd.read_parquet(iri.output_dir("r1_q1") / "de_results.parquet")

# 4. Sanity-check what loaded.
print(f"Loaded: {de_results.shape}")
print(f"Pairs:  {de_results.groupby(['stress', 'tissue']).ngroups}")
de_results.head()

## 1) Collapsing to per-stress DE flags

The DE table from last week has one row per (gene, stress, tissue) — 364,960 rows across eight stresses and two tissues. To ask "in how many stresses is this gene DE?" we first need to decide what "DE for a stress" means when each stress has two tissue measurements.

Last week's Section 8 showed that root and shoot responses can differ substantially within a stress, and the AtGenExpress design itself is asymmetric: three whole-plant stresses (cold, drought, heat) directly stress both tissues, while five locally-applied stresses have one directly-stressed tissue and one systemic-response tissue. Requiring DE in *both* tissues per stress would penalize the local-application stresses for their experimental design — a gene clearly responding to wounding in the wounded tissue could fail to count just because the systemic tissue is quiet.

The choice we're making here: **a gene is DE for a stress if it's DE in either tissue.** This is the more permissive of the natural options, and it handles the tissue asymmetry without forcing us to model it explicitly. The cost is information loss — we collapse two bits per (gene, stress) down to one — but the per-tissue detail is preserved in the original `de_results` table for anything downstream that needs it.

Two steps:

1. Collapse the tissue dimension. Group the long-format table by (gene, stress) and take the OR across tissues. Produces a table of `(gene, stress, is_de_either_tissue)`.
2. Count stresses per gene. Group the collapsed table by gene and sum the boolean column. Produces a per-gene count from 0 to 8 — how many stresses each gene responds to.

This section picks up that per-gene count and asks where to draw the threshold N.

In [ ]:
# Collapse to per-stress DE flags: a gene is DE for a stress
# if it's DE in either tissue.
de_per_stress = (
    de_results
    .groupby(["gene", "stress"], as_index=False)["is_de"]
    .any()
    .rename(columns={"is_de": "is_de_either_tissue"})
)

print(f"Rows: {len(de_per_stress):,}")
print(f"Genes: {de_per_stress['gene'].nunique():,}")
print(f"Stresses: {de_per_stress['stress'].nunique()}")
de_per_stress.head()

In [ ]:
# Count stresses per gene: how many of the 8 did it respond to?
stresses_per_gene = (
    de_per_stress
    .groupby("gene")["is_de_either_tissue"]
    .sum()
)

print(f"Genes:    {len(stresses_per_gene):,}")
print(f"Range:    {stresses_per_gene.min()} to {stresses_per_gene.max()}")
print(f"Mean:     {stresses_per_gene.mean():.2f}")
print(f"Median:   {stresses_per_gene.median():.0f}")
stresses_per_gene.head()

## 2) How many genes pass at each threshold N?

We pre-committed to N=6 — a gene counts toward the core if it's DE in at least six of the eight stresses — on the grounds that "responds to a supermajority of stresses" is what we'd want from a *core* responder. That commitment was made before looking at the data, which is the point: the choice shouldn't depend on which N produces the nicest-looking number.

Before applying N=6, this section looks at the full count-vs-N distribution. Two reasons. First, the shape of the distribution is itself informative — it tells us whether genes cluster at one end (most don't respond to anything), the other end (most respond to everything), or somewhere in the middle. Second, the position of N=6 on that distribution determines how big the core is, which we'll need for the comparison with Hakkak & Tohidfar 2026 in Notebook 4 and for the enrichment analysis later in this notebook.

A note on what you're about to see: the distribution is unusual. Most cross-stress analyses produce a heavily right-skewed shape — many genes respond to few or no stresses, a thin tail responds to many. The AtGenExpress count-vs-N distribution at our working DE threshold is closer to U-shaped, with peaks at *both* ends. That shape is a real finding and worth thinking about; we come back to it in the interpretation below and the threshold question becomes one of the substrates EQ#2 will work through.

In [ ]:
import matplotlib.pyplot as plt

# Build the count table: N stresses → number of genes.
counts = stresses_per_gene.value_counts().sort_index()

# Pre-committed threshold.
N_COMMIT = 6
core_size = counts.loc[N_COMMIT:].sum()

# Color bars: grey for non-core, blue for N >= N_COMMIT.
colors = ["#b3b3b3" if n < N_COMMIT else "#3182bd" for n in counts.index]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(counts.index, counts.values, color=colors,
              edgecolor="black", linewidth=0.4)

# Numeric label on top of each bar.
for bar, value in zip(bars, counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 60,
        f"{value:,}",
        ha="center", va="bottom", fontsize=9,
    )

ax.set_xlabel("Number of stresses the gene is DE in (out of 8)")
ax.set_ylabel("Number of genes")
ax.set_title(
    f"Genes by number of stresses they respond to  "
    f"(core, N ≥ {N_COMMIT}: {core_size:,} genes)"
)
ax.set_xticks(counts.index)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_ylim(0, counts.max() * 1.10)

plt.tight_layout()
plt.show()

print(f"Core size at N >= {N_COMMIT}: {core_size:,} genes")

## 3) What this distribution tells us

The shape of the plot above is unusual. Most cross-stress analyses produce a heavily right-skewed distribution: most genes don't respond to most stresses, and a thin tail of cross-stress responders sits at the high end. The chart above shows something different — a U-shape, with peaks at *both* ends and a quieter middle. There are 2,331 genes that respond to no stress at all, 4,888 genes that respond to every one of the eight, and a trough of around 1,800 genes at the supermajority range we pre-committed to.

That right-end peak is the immediate surprise. Almost 5,000 genes — roughly 21% of the array — are flagged as differentially expressed in cold AND drought AND heat AND osmotic AND salt AND genotoxic AND UV-B AND wounding. The core at N ≥ 6 is 9,067 genes, about 40% of the array.

The R1-Q1 page predicted a core "on the order of hundreds." The result is two orders of magnitude larger than that. This is a real finding rather than a problem to fix — the data is the data, the pre-commitment was made on principled grounds before we looked, and the gap between prediction and result is itself informative. But the gap is large enough to demand an explanation.

### 3.1) A likely explanation - Permissive threshold


The most likely explanation traces back to last week's notebook. The working DE threshold there was |logFC| ≥ 1 — the lower end of the conventional microarray cutoffs. Section 5 of that notebook surfaced a related finding: every gene clearing |logFC| ≥ 1 also cleared the adjusted-p-value filter at 0.05, meaning the statistical-significance test was redundant rather than additive. That was a signal, retrospectively, that the threshold was permissive — it was admitting too many genes whose expression changes were technically detectable but biologically modest. A permissive per-pair threshold compounds across stresses: a gene that crosses a low bar in each of eight independent experiments will appear in the cross-stress intersection at much higher rates than a gene crossing a meaningful bar would. The U-shape's right peak is, on this reading, the population of "always changing" genes that a stricter cutoff would compress.

### 3.2) Follow-up
A natural follow-up would be to rerun last week's differential expression at a stricter cutoff — |logFC| ≥ 2 is the conventional next step — and recompute this distribution. We're not doing that here. The pre-commitment to N=6 was made on principled grounds and the prediction-versus-result gap is part of what this notebook is about; tightening the upstream threshold now would be exactly the kind of post-hoc adjustment the pre-commitment exists to prevent. The threshold question is the right question, but it belongs to a follow-up iteration, not to this notebook.

For the analysis that follows, we use the 9,067-gene core as defined: genes DE in at least 6 of 8 stresses, with "DE for a stress" meaning DE in either tissue. Functional enrichment in the next section asks what *kind* of genes these are — whether the core, large as it is, is dominated by the families a stress core would be expected to contain (ROS metabolism, heat shock, protein folding, broadly acting transcription factors), or whether the inflation has pulled in genes from unrelated functional categories.

## 4) Building and saving the core

The pre-commitment was N=6 and "either tissue counts." Applying that to the per-gene stress counts gives the core gene set: 9,067 genes that respond to at least six of the eight stresses, in at least one tissue per stress.

The artifact we save is a two-column table — `gene` and `n_stresses` — with one row per core member. Including the count per gene gives downstream notebooks the option to weight or sort by it without having to recompute. The file lands in Drive at the per-question output directory, matching the convention last week's notebook established for the differential expression table.

In [ ]:
# Build the core: genes responding to at least N_COMMIT stresses.
core = (
    stresses_per_gene[stresses_per_gene >= N_COMMIT]
    .rename("n_stresses")
    .reset_index()
    .sort_values("n_stresses", ascending=False)
    .reset_index(drop=True)
)

print(f"Core size:    {len(core):,} genes")
print(f"N=8 members:  {(core['n_stresses'] == 8).sum():,}")
print(f"N=7 members:  {(core['n_stresses'] == 7).sum():,}")
print(f"N=6 members:  {(core['n_stresses'] == 6).sum():,}")
core.head()

The core now exists in memory as `core`. To make it available to Notebook 4 — which will compare this set to the Hakkak & Tohidfar 2026 consensus core — we save it as parquet in the per-question output directory, following the same path convention last week's notebook used for the DE table.

In [ ]:
# Save the core to Drive for Notebook 4 to consume.
core_path = iri.output_dir("r1_q1") / "core_genes.parquet"
core.to_parquet(core_path)
print(f"Saved: {core_path}")
print(f"Size:  {core_path.stat().st_size / 1024:.1f} KB")

# Confirm round-trip by reading back and checking shape.
import pandas as pd
core_check = pd.read_parquet(core_path)
assert core_check.shape == core.shape, "Saved file shape doesn't match in-memory core"
print(f"Round-trip OK: {core_check.shape}")

## 5) Functional enrichment of the core
Section 3 ended with a question: what *kind* of genes are these 9,067? The size is much larger than the R1-Q1 page predicted, and the question now is whether the core is dominated by stress-related gene families — ROS metabolism, heat shock, protein folding, broadly acting transcription factors — or whether the threshold inflation has pulled in genes from unrelated categories.

The standard tool for this is **functional enrichment**, specifically over-representation analysis (ORA). Each genome gene is annotated with GO terms describing its role. ORA asks: is a GO term more common in the core than expected by chance? The test uses hypergeometric statistics — the same machinery as drawing colored balls from an urn without replacement.

Before running, two choices matter. First, the tool: we use `goatools`, a Python-based tool with local GO files, producing consistent results upon rerun. Second, the background set: the natural choice is all *Arabidopsis* genes (~27,000), but our core was drawn from 22,810 genes on the ATH1 microarray. Using the full genome inflates significance for GO terms favored by the microarray design. Instead, the correct background is the 22,810 genes we used, representing what could have been selected.

One preliminary step. Our core is keyed on ATH1 probe IDs (the `gene` column uses Affymetrix probe identifiers like `244901_at`), but GO annotations are keyed on AGI codes (the *Arabidopsis* Genome Initiative gene identifiers like `AT1G01010`). The two systems don't overlap, so we translate probes to AGI codes before running enrichment.

The analysis ranks GO terms by their over-representation in the core. A core that *is* stress-related will show GO terms recognizable as stress biology. A core that *isn't* will show terms that are either generic or unrelated.

We start with the biological process (BP) branch, most relevant for the stress-response question we're asking. Molecular function (MF) and cellular component (CC) can follow if BP results are inconclusive.

In [ ]:
import GEOparse

gpl = GEOparse.get_GEO(geo='GPL198', destdir=str(iri.cache_dir()), silent=True)

print(f"Annotation table: {gpl.table.shape}")
gpl.table[['ID', 'AGI']].head()

In [ ]:
# Build a probe -> AGI mapping from the GPL198 annotation table.
#
# Two cases to handle defensively:
#   1. Multi-locus probes: a probe may target multiple AGI loci, separated
#      by ' /// '. We take the first locus (the probe's primary target).
#   2. Missing AGI: some probes have no AGI annotation at all (empty or NaN).
#      Those probes can't be enrichment-analyzed and get dropped.
probe_to_agi = {}
for _, row in gpl.table[['ID', 'AGI']].dropna().iterrows():
    probe = row['ID']
    first_locus = str(row['AGI']).split(' /// ')[0].strip().upper()
    if first_locus:
        probe_to_agi[probe] = first_locus

print(f"Probes with AGI mapping:  {len(probe_to_agi):,}")
print(f"Probes without AGI:       {22810 - len(probe_to_agi):,}")

# Translate the core (9,067 probes -> AGI codes).
core_agi = (
    core['gene']
    .map(probe_to_agi)
    .dropna()
    .unique()
    .tolist()
)

# Translate the background (22,810 probes -> AGI codes).
# The background is the full set of genes we ran DE on, which is every
# probe in the de_results table.
background_agi = (
    de_results['gene']
    .drop_duplicates()
    .map(probe_to_agi)
    .dropna()
    .unique()
    .tolist()
)

print()
print(f"Core (AGI):       {len(core_agi):,} unique loci")
print(f"Background (AGI): {len(background_agi):,} unique loci")

Two reference files are needed. `go-basic.obo` is the GO ontology itself — the directed acyclic graph of terms and their parent–child relationships. The gene-to-GO mapping (`tair.gaf`) is what associates each *Arabidopsis* gene with its annotated GO terms. Both are external downloads on first run; `goatools` caches them locally afterward, so subsequent runs are network-free.

In [ ]:
import gzip, shutil
from goatools.base import download_go_basic_obo

# OBO: download from upstream (this worked last run).
obo_path = iri.cache_dir() / "go-basic.obo"
if not obo_path.exists():
    download_go_basic_obo(obo=str(obo_path))
else:
    print(f"OBO already cached: {obo_path}")

# GAF: decompress from the bundled file in the library.
gaf_path = iri.cache_dir() / "tair.gaf"
if not gaf_path.exists():
    with gzip.open(iri.tair_gaf_path(), 'rb') as f_in, open(gaf_path, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    print(f"GAF extracted: {gaf_path}")
else:
    print(f"GAF already cached: {gaf_path}")

print(f"\nOBO: {obo_path.stat().st_size / 1024 / 1024:.1f} MB")
print(f"GAF: {gaf_path.stat().st_size / 1024 / 1024:.1f} MB")